# 02 · Feature Engineering
**Notebook 2 of 4** · run order: 01 → 02 → 03 → 04

> Synthetic demonstration data. See notebook 01 for what that means.

**What this notebook does**
1. Choose who we model and which months we may look at
2. Turn monthly rows into one row of features per customer
3. Check the features for problems (missing values, leakage)
4. Split into train / validation / test, **stratified** so every type of customer is represented

**Reads:** `data/panel_monthly.parquet`, `work/labels.parquet`, `work/clean_customers.parquet`
**Saves:** `work/train.parquet`, `work/valid.parquet`, `work/test.parquet`, `work/figures/feature_vs_label.png`

## 0 · Setup

**Where to run it**
- **Your computer:** keep the folder layout as delivered (`data/` next to the notebooks). Nothing to change.
- **Google Colab:** upload the two parquet files, set `DATA_DIR = "/content"` and `WORK_DIR = "/content/work"`.
- **Kaggle:** add the parquet files as a dataset, set `DATA_DIR = "/kaggle/input/<your-dataset-name>"` and `WORK_DIR = "/kaggle/working"`.

In [ ]:
# ---- Settings: change these two lines if you run on Colab or Kaggle ----
DATA_DIR = "data"   # folder that holds panel_monthly.parquet and customers.parquet
WORK_DIR = "work"   # folder where the notebooks save their results for the next notebook

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

FIG_DIR = os.path.join(WORK_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
LABEL = "value_compressed_flag"

for name in ["labels.parquet", "clean_customers.parquet"]:
    if not os.path.exists(os.path.join(WORK_DIR, name)):
        print("Missing", name, "- please run notebook 01 first.")

In [ ]:
panel = pd.read_parquet(os.path.join(DATA_DIR, "panel_monthly.parquet"))
labels = pd.read_parquet(os.path.join(WORK_DIR, "labels.parquet"))
customers = pd.read_parquet(os.path.join(WORK_DIR, "clean_customers.parquet"))
print("panel:", panel.shape, "| labels:", labels.shape, "| customers:", customers.shape)

## 1 · Who do we model, and which months may we use?
- **Who:** active customers only (they have not left, so we can still act on them).
- **Which months:** only **months 1–9**. The label looks at the last three months (10–12), so features must come from *before* that. Using later months would be cheating.

```
Month:    1  2  3  4  5  6  7  8  9 | 10  11  12
          [------ features -------]   [ label window ]
```

In [ ]:
active_ids = labels[labels["churned_flag"] == 0].index
observed = panel[(panel["month"] <= 9) & (panel["customer_id"].isin(active_ids))]

print("Active customers:", len(active_ids))
print("Rows used for features:", len(observed))
print("Months used:", sorted(observed["month"].unique().tolist()))

## 2 · Build the features
For each of the four behaviour columns we compute, per customer:

| Feature | Meaning |
|---|---|
| `_mean` | average over months 1–9 |
| `_first3` | average over months 1–3 (the customer's early normal) |
| `_last3` | average over months 7–9 (most recent behaviour) |
| `_max` | worst single month |
| `_change` | `_last3` minus `_first3` (is it getting worse?) |

The `_change` features are a simple **trend over time**. Revenue is not used.

In [ ]:
behaviours = ["monthly_overage_fee", "data_overage_fee",
              "dropped_blocked_calls", "customer_service_calls"]

average = observed.groupby("customer_id")[behaviours].mean().add_suffix("_mean")
first3 = observed[observed["month"] <= 3].groupby("customer_id")[behaviours].mean().add_suffix("_first3")
last3 = observed[observed["month"] >= 7].groupby("customer_id")[behaviours].mean().add_suffix("_last3")
worst = observed.groupby("customer_id")[behaviours].max().add_suffix("_max")

features = pd.concat([average, first3, last3, worst], axis=1)
features.head()

In [ ]:
for column in behaviours:
    features[column + "_change"] = features[column + "_last3"] - features[column + "_first3"]

print("Behaviour features so far:", features.shape[1])

### Add facts that do not change over time

In [ ]:
# tenure at month 9
features["tenure_months"] = observed[observed["month"] == 9].set_index("customer_id")["account_tenure_months"]

# income and the missing-income flag
facts = customers.set_index("customer_id")[["estimated_income_bracket", "income_missing"]]
features = features.join(facts)

print("Total features:", features.shape[1])
list(features.columns)

### Attach the label

In [ ]:
data = features.join(labels[LABEL])
print("Modelling table:", data.shape)
print("Compressed share:", round(data[LABEL].mean() * 100, 1), "%")
data.head()

## 3 · Quality checks

In [ ]:
print("Missing values:", data.isna().sum().sum())
print("Duplicate customers:", data.index.duplicated().sum())

forbidden = [c for c in features.columns
             if "revenue" in c or "latent" in c or "churn" in c or "segment" in c]
print("Columns that must NOT be features (revenue, answer key, churn, segment):", forbidden)

### Which features move with the label?
A quick look before modelling. Correlation only shows a straight-line link; the model can find more.

In [ ]:
corr = data.corr()[LABEL].drop(LABEL)
strongest = corr.reindex(corr.abs().sort_values().tail(12).index)

plt.figure(figsize=(8, 5))
strongest.plot(kind="barh", color="#B45309")
plt.title("12 features most correlated with the label")
plt.xlabel("Correlation with value_compressed_flag")
plt.savefig(os.path.join(FIG_DIR, "feature_vs_label.png"), dpi=130, bbox_inches="tight")
plt.show()

## 4 · Stratified split
**Why stratified?** Only a minority of customers are compressed, and customer types differ (new vs old, income level, unknown income). A plain random split can leave some types under-represented in one part. Stratifying keeps the **same mix of customer types in every part**.

We stratify on the label combined with an income group and a tenure band.

In [ ]:
income_group = pd.cut(data["estimated_income_bracket"], bins=[-2, 0, 2, 4, 6],
                      labels=["missing", "low", "mid", "high"])
tenure_band = pd.cut(data["tenure_months"], bins=[0, 24, 48, 200],
                     labels=["new", "mid", "long"])

strata = data[LABEL].astype(str) + "_" + income_group.astype(str) + "_" + tenure_band.astype(str)
print("Number of customer types (strata):", strata.nunique())
strata.value_counts().head(6)

In [ ]:
# 1) hold out 30% for the final test
train_valid, test = train_test_split(data, test_size=0.30, stratify=strata, random_state=42)

# 2) split the rest into train (80%) and validation (20%) - validation is used for tuning in notebook 03
train, valid = train_test_split(train_valid, test_size=0.20,
                                stratify=strata[train_valid.index], random_state=42)

print("train:", len(train), "| valid:", len(valid), "| test:", len(test))

### Check: is every customer type represented the same way in each part?

In [ ]:
rows = []
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    rows.append({
        "set": name,
        "customers": len(part),
        "compressed %": round(part[LABEL].mean() * 100, 2),
        "income unknown %": round((income_group[part.index] == "missing").mean() * 100, 2),
        "long tenure %": round((tenure_band[part.index] == "long").mean() * 100, 2),
    })
pd.DataFrame(rows)

In [ ]:
overlap = (set(train.index) & set(valid.index)) | (set(train.index) & set(test.index)) | (set(valid.index) & set(test.index))
print("Customers that appear in more than one set:", len(overlap))

### Save for notebook 03

In [ ]:
train.to_parquet(os.path.join(WORK_DIR, "train.parquet"))
valid.to_parquet(os.path.join(WORK_DIR, "valid.parquet"))
test.to_parquet(os.path.join(WORK_DIR, "test.parquet"))
print("Saved train, valid and test.")

## 5 · Summary
- 23 features were built from months 1-9 only. Revenue (which defines the label) is not used, and the generator's answer key was removed in notebook 01.
- Active customers were split into train, validation and test sets, stratified on label, income group and tenure band (24 customer types). The table in section 4 shows the mix is almost identical in all three.
- No customer appears in more than one set.

**Next:** `03_model_training_and_testing.ipynb`